# Executive Summary ETL

## Purpose
Provide a single-row executive dashboard with high-level KPIs for business leadership.

## Input → Output
* **Source 1:** `big_data.silver.orders` 
* **Source 2:** `big_data.silver.order_products` 
* **Source 3:** `big_data.silver.products_enriched` 
* **Target:** `big_data.gold.ft_executive_summary` 
* **Primary Key:** None (single aggregated row)

## Transformations
1. Load Silver Tables and Join - Load orders, products, order_products and join order_products with products for pricing
2. Calculate Executive KPIs - Aggregate total_orders, total_customers, total_products, total_items_sold, estimated_total_revenue, avg_basket_size, avg_order_value, overall_reorder_rate, add timestamp

## Data Quality
* **Technical:** Row count = 1, NOT NULL (all KPIs), All numeric values >= 0
* **Business:** Total orders > 3M, Total customers > 200K, Reorder rate 50-70%, Avg basket size 8-12 items

## Persistence
Writes to Delta table **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType

In [0]:
# Schema configuration
silver_schema = "big_data.silver"
gold_schema = "big_data.gold"

# Source tables
source_tables = {
    "orders": "orders",
    "products": "products_enriched",
    "order_products": "order_products"
}

# Target table (fact table with ft_ prefix)
target_table = "ft_executive_summary"

# Primary Key columns (none for aggregated table)
primary_key_columns = []  # Single aggregated row, no PK

# Critical columns (all KPIs must be NOT NULL)
critical_columns = [
    "total_orders", "total_customers", "total_products", 
    "total_items_sold", "estimated_total_revenue_usd",
    "avg_basket_size", "avg_order_value_usd", "overall_reorder_rate"
]

# Expected metrics (for validation)
expected_metrics = {
    "min_total_orders": 3_000_000,
    "min_total_customers": 200_000,
    "min_reorder_rate": 50.0,
    "max_reorder_rate": 70.0,
    "min_avg_basket_size": 8.0,
    "max_avg_basket_size": 12.0
}

# Create Gold schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Configuration:")
print(f"  Source schema: {silver_schema}")
print(f"  Target: {gold_schema}.{target_table}")
print(f"  Source tables: {list(source_tables.keys())}")

### TRANSFORMATION

In [0]:
print("Step 1: Loading Silver tables and joining...")

# Load Silver tables
orders = spark.table(f"{silver_schema}.{source_tables['orders']}")
products = spark.table(f"{silver_schema}.{source_tables['products']}")
order_products = spark.table(f"{silver_schema}.{source_tables['order_products']}")

print(f"  Orders loaded: {orders.count():,} rows")
print(f"  Products loaded: {products.count():,} rows")
print(f"  Order-Products loaded: {order_products.count():,} rows")

# Join order_products with products to get pricing
order_products_enriched = order_products \
    .join(products.select("product_id", "price_usd"), "product_id", "left")

print(f"\n  Enriched order_products with pricing: {order_products_enriched.count():,} rows")

In [0]:
print("Step 2: Calculating executive KPIs...")

# Calculate order-level metrics first
order_level_metrics = orders \
    .join(order_products_enriched, "order_id") \
    .groupBy("order_id") \
    .agg(
        F.count("product_id").alias("basket_size"),
        F.round(F.sum("price_usd"), 2).alias("estimated_order_value_usd"),
        F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)).alias("reordered_items")
    )

# Calculate executive summary
executive_summary_df = order_level_metrics.agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.avg("basket_size"), 2).alias("avg_basket_size"),
    F.round(F.avg("estimated_order_value_usd"), 2).alias("avg_order_value_usd"),
    F.round(F.sum("estimated_order_value_usd"), 2).alias("estimated_total_revenue_usd"),
    F.sum("reordered_items").alias("total_reordered_items")
)

# Add global metrics
total_customers = orders.select("user_id").distinct().count()
total_products = products.select("product_id").distinct().count()
total_items = order_products.count()

executive_summary_gold = executive_summary_df \
    .withColumn("total_customers", F.lit(total_customers)) \
    .withColumn("total_products", F.lit(total_products)) \
    .withColumn("total_items_sold", F.lit(total_items)) \
    .withColumn(
        "overall_reorder_rate",
        F.round(
            (F.col("total_reordered_items") / F.col("total_items_sold")) * 100,
            2
        )
    ) \
    .withColumn("_gold_timestamp", F.current_timestamp())

print("  Executive KPIs calculated")
print("\nPreview:")
executive_summary_gold.show(1, truncate=False, vertical=True)

In [0]:
# Create final DataFrame for validation and persistence
df_result = executive_summary_gold

print(f"\nFinal DataFrame 'df_result' created: {df_result.count():,} row")
print("\nReady for validation and persistence")

### DATA QUALITY

In [0]:
print_validation_header("Executive Summary - Technical Validations")

total_rows = df_result.count()
print(f"\nTotal rows: {total_rows:,}")
print(f"Expected: 1 row (aggregated summary)\n")

# Initialize validation flag
validation_technical = True

# 1. Row count check (must be exactly 1)
if total_rows != 1:
    status = "FAIL"
    msg = f"Expected 1 row, got {total_rows}"
    validation_technical = False
else:
    status = "PASS"
    msg = "Exactly 1 aggregated row"
print_check_result("ROW COUNT (=1)", status, msg)

# 2. NOT NULL checks (all KPI columns)
print("\n2. NOT NULL Validations:")
status, failed, msg = check_not_null(df_result, critical_columns)
print_check_result(f"NOT NULL ({len(critical_columns)} KPIs)", status, msg, failed)
if status == "FAIL":
    validation_technical = False

# 3. Non-negative checks (all metrics should be >= 0)
print("\n3. Non-negative Validation:")
negative_count = 0
for col_name in critical_columns:
    col_value = df_result.select(col_name).first()[0]
    if col_value is not None and col_value < 0:
        print(f"  ✗ {col_name}: {col_value} (NEGATIVE)")
        negative_count += 1
        validation_technical = False

if negative_count == 0:
    status = "PASS"
    msg = "All metrics are non-negative"
else:
    status = "FAIL"
    msg = f"{negative_count} metrics are negative"
print_check_result("NON-NEGATIVE (all KPIs >= 0)", status, msg, negative_count)

print("\n" + "="*60)
if validation_technical:
    print("SUCCESS: Technical validations PASSED")
else:
    print("FAILURE: Technical validations FAILED")
print("="*60)

In [0]:
print_validation_header("Executive Summary - Business Validations")

# Initialize business validation flag
validation_business = True

# Get the summary row
summary_row = df_result.first()

# 1. Total orders validation
print("\n1. Business Rule - Total Orders:")
total_orders = summary_row["total_orders"]
if total_orders >= expected_metrics["min_total_orders"]:
    status = "PASS"
    msg = f"Total orders ({total_orders:,}) >= {expected_metrics['min_total_orders']:,}"
else:
    status = "FAIL"
    msg = f"Total orders ({total_orders:,}) < {expected_metrics['min_total_orders']:,}"
    validation_business = False
print_check_result("TOTAL ORDERS (>= 3M)", status, msg)

# 2. Total customers validation
print("\n2. Business Rule - Total Customers:")
total_customers = summary_row["total_customers"]
if total_customers >= expected_metrics["min_total_customers"]:
    status = "PASS"
    msg = f"Total customers ({total_customers:,}) >= {expected_metrics['min_total_customers']:,}"
else:
    status = "FAIL"
    msg = f"Total customers ({total_customers:,}) < {expected_metrics['min_total_customers']:,}"
    validation_business = False
print_check_result("TOTAL CUSTOMERS (>= 200K)", status, msg)

# 3. Reorder rate validation
print("\n3. Business Rule - Reorder Rate:")
reorder_rate = summary_row["overall_reorder_rate"]
min_rate = expected_metrics["min_reorder_rate"]
max_rate = expected_metrics["max_reorder_rate"]
if min_rate <= reorder_rate <= max_rate:
    status = "PASS"
    msg = f"Reorder rate ({reorder_rate}%) within expected range [{min_rate}%, {max_rate}%]"
else:
    status = "FAIL"
    msg = f"Reorder rate ({reorder_rate}%) outside expected range [{min_rate}%, {max_rate}%]"
    validation_business = False
print_check_result("REORDER RATE (50-70%)", status, msg)

# 4. Avg basket size validation
print("\n4. Business Rule - Avg Basket Size:")
avg_basket = summary_row["avg_basket_size"]
min_basket = expected_metrics["min_avg_basket_size"]
max_basket = expected_metrics["max_avg_basket_size"]
if min_basket <= avg_basket <= max_basket:
    status = "PASS"
    msg = f"Avg basket size ({avg_basket}) within expected range [{min_basket}, {max_basket}]"
else:
    status = "FAIL"
    msg = f"Avg basket size ({avg_basket}) outside expected range [{min_basket}, {max_basket}]"
    validation_business = False
print_check_result("AVG BASKET SIZE (8-12)", status, msg)

print("\n" + "="*60)
if validation_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta table using UTILS function
if validation_passed:
    persist_to_delta(df_result, f"{gold_schema}.{target_table}")
    print("\nNext Step: Query for executive insights and KPI monitoring")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")